In [0]:
from pyspark.sql.types import StructType, StructField,StringType, IntegerType, DateType, TimestampType, FloatType 

import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

###Read and Vakidate Bronze brands table

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands");
df_bronze.show(10)

### Silver Clean Data Frame

In [0]:
df_silver = df_bronze.withColumn("brand_name",F.trim(F.col("brand_name")));
df_silver.limit(5).show()

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"),r"[^A-Za-z0-9]",""))
df_silver.limit(10).show()

In [0]:
df_silver.select("category_code").distinct().show()


In [0]:
#Anomalies Dictionary
anomalies ={
    "GROCERY" : "GRCY",
    "BOOKS" : "BKS",
    "TOYS" : "TOY"
}

df_silver = df_silver.replace(anomalies, subset="category_code")


In [0]:
df_silver.select("category_code").distinct().show()

###Drop the table in spark

In [0]:
spark.sql(f"Drop table {catalog_name}.silver.slv_brands")

In [0]:
df_silver.printSchema()

In [0]:
#Writing raw_data into the silver layer (catalog name : ecommerce, schema name: silver, table name : slv_brands)

df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

#Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show(10)

### Finding Duplicate Records

In [0]:
df_duplicate = df_bronze.groupBy("category_code").count().filter(F.col("count")>1)
display(df_duplicate)

###Clearing Duplicate Records

In [0]:
df_silver =df_bronze.dropDuplicates(["CATEGORY_CODE"])
display(df_silver)

###Coverting to UpperCase

In [0]:
df_silver = df_silver.withColumn("category_code",F.upper(F.col("category_code")))
display(df_silver)

###Create Delta Table - slv_category

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema", "true")\
            .saveAsTable(f"{catalog_name}.silver.slv_category");

#Prducts

In [0]:
#Read the raw data from the bronze table (ecommerce.bronze.brz_products)
df_bronze = spark.table(f"{catalog_name}.bronze.brz_products")
display(df_bronze)

###Counting Rows and Columns

In [0]:
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

print(f"Rount Count is :{row_count}")
print(f"Column Count is : {column_count}")

In [0]:
#Check weigh_grams column

df_bronze.select("weight_grams").show(5, truncate = False)

###Replacing G with ''

In [0]:
df_silver = df_bronze.withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"),"g","").cast(IntegerType()))

In [0]:
df_silver.select("weight_grams").show(5, truncate=False)

###Check length_cm

In [0]:
df_silver.select("length_cm").show(10, truncate=False)

### Replacing dot instead of Comma

In [0]:
df_silver= df_silver.withColumn("length_cm",F.regexp_replace(F.col("length_cm"),",",".").cast(FloatType()))

In [0]:
display(df_silver.select("length_cm").show(5))

In [0]:
df_silver.select("category_code","brand_code").show()

###Coverting to uppercase Two coluns such as category_code and brand_code

In [0]:
df_silver = df_silver.withColumn("category_code",F.upper(F.col("category_code"))).withColumn("brand_code",F.upper(F.col("brand_code")))

In [0]:
df_silver.select("category_code","brand_code").show()


###Fixing speelling mistakes in materialcolumn

In [0]:
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material")=="Coton","Cotton")
    .when(F.col("material")=="Alumium","Aluminum")
    .when(F.col("material")=="Ruber","Rubber")
    .otherwise(F.col("material"))
        )

df_silver.select("material").show()

###negative values in rating_count

In [0]:
df_silver.filter(F.col("rating_count")<1).select("rating_count").show()

###Converting megative rating_count to positive 

In [0]:
df_silver= df_silver.withColumn("rating_count",F.when(
    F.col("rating_count").isNotNull(),
    F.abs(F.col("rating_count")))
    .otherwise(F.lit(0))  #If null, replace with 0
)

In [0]:
display(df_silver.select("weight_grams","length_cm","category_code","brand_code","material","rating_count").show(10, truncate = False)) 

###Create Delta Table - slv_products

In [0]:
#Inseting raw data into the silver layer (catalog : ecommerce, schema name : silver, table name : products)

df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_products")


#Customer - Reading customer table from 
bronze layer

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_customers")
df_bronze.show()

In [0]:
df_bronze=df_bronze.withColumn("country",\
    F.when(F.col("country")=="India","Myanmar")\
    .otherwise(F.col("country")
     ) )
df_bronze.show()

### Handle Null Values in Customer_Id Column

In [0]:
null_count = df_bronze.filter(F.col("customer_id").isNull()).count()
null_count

In [0]:
df_bronze.filter(F.col("customer_id").isNull()).show(3)

Dropping rows once the customerid is null

In [0]:
df_silver = df_bronze.dropna(subset=["customer_id"])


Geeting row count

In [0]:
row_count = df_silver.count()
print(f"Row count after dropping null values : {row_count}")